# Predict masked CDR3 residues

Load the model, mask a known CDR3 span, and see what LangAAI predicts there given the antibody's framework and its antigen.

In [ ]:
import langaai

model = langaai.load()
print(model.device, model.dim)

A real heavy/light pair and its cognate antigen. Variable-domain sequences are what the model expects -- not the full chain including the constant region.

In [ ]:
heavy = "EVQLVESGGGLVQPGGSLRLSCAASGFNFKDTYIHWVRQAPGKGLEWVARIYPANGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCASDGGSYSYAFDYWGQGTLVTVSS"
light = "DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIK"
antigen = "TQVCTGTDMKLRLPASPETHLDMLRHLYQGCQVVQGNLELTYLPTNASLSFLQDIQEVQGYVLIAHNQVRQVPLQRLRIVRGTQLFEDNYALAVLDNGDPLNNTTPVTGASPGGLRELQLRSLTEILKGGVLIQRNPQLCYQDTILWKDIFHKNNQLALTLIDTNRSRACHPCSPMCKGSRCWGESSEDCQSLTRTVCAGGCARCKGPLPTDCCHEQCAAGCTGPKHSDCLACLHFNHSGICELHCPALVTYNTDTFESMPNPEGRYTFGASCVTACPYNYLSTDVGSCTLVCPLHNQEVTAEDGTQRCEKCSKPCARVCYGLGMEHLREVRAVTSANIQEFAGCKKIFGSLAFLPESFDGDPASNTAPLQPEQLQVFETLEEITGYLYISAWPDSLPDLSVFQNLQVIRGRILHNGAYSLTLQGLGISWLGLRSLRELGSGLALIHHNTHLCFVHTVPWDQLFRNPHQALLHTANRPEDECVGEGLACHQLCARGHCWGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYVNARHCLPCHPECQPQNGSVTCFGPEADQCVACAHYKDPPFCVARCPSGVKPDLSYMPIWKFPDEEGACQPCPIN"
print(len(heavy), len(light), len(antigen))

ab = model.encode_antibody(heavy, light)
ag = model.embed_antigen(antigen)
print(f"antibody: {len(ab)} tokens, antigen: {len(ag)} residues (truncated={ag.truncated})")

CDR3 (`ASDGGSYSYAFDY`, this heavy chain's known CDR3) sits at positions 96-108 -- found by substring search here for the notebook, but in general spans always come from your own upstream annotation. `mask_region` never computes CDR boundaries itself.

In [ ]:
cdr3 = "ASDGGSYSYAFDY"
start = heavy.find(cdr3)
span = (start, start + len(cdr3))
print("CDR3 span:", span)

masked = ab.mask_region("cdr3", spans=[span])
[predictions] = model.predict_masked([(masked, ag)], top_k=5)
for p, true_letter in zip(predictions, cdr3):
    top1 = p.top[0]
    hit = "OK" if top1[0] == true_letter else "  "
    print(f"pos {p.position:3d}  true={true_letter}  top1={top1[0]} ({top1[1]:.2f}) [{hit}]  top5={p.top}")

`score_sequence` gives one number for how well the whole masked span fits, given the antigen -- useful for comparing candidate CDR3s against each other, **not** a validated affinity predictor.

In [ ]:
positions = list(range(*span))
[score] = model.score_sequence([(ab, ag)], positions=[positions])
print(f"mean log-likelihood over CDR3 given the true antigen: {score:.3f}")